<header>
   <p style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
        Getting Started with Embedding-Based Collection
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

## Introduction

This notebook demonstrates how to work with **Embedding-Based Collection** using pre-computed embeddings.

Collections provide a stateless API for managing vector stores with existing embeddings. This is useful when you already have embeddings generated from another system or want to use custom embeddings.

### What You'll Learn

1. Setting up your environment and connecting to Vantage
2. Loading example data with pre-computed embeddings (1024 dimensions)
3. Initializing embedding and chat models
4. Creating an embedding-based collection using the `create()` method
5. Retrieving collection details and metadata
6. Updating collections using the `update()` method (adding/deleting datasets)
7. Performing similarity search and asking questions (with embedding model)
8. Cleaning up resources

---
## 1. Setup

### Install teradatagenai

First, ensure you have the latest version of teradatagenai installed.

In [ ]:
# %pip install teradatagenai --upgrade

### Import Required Modules

In [1]:
import os
import time
from getpass import getpass

from teradatagenai import Collection, TeradataAI, load_data, EmbeddingBasedIndex, ColumnInfo, CollectionManager
from teradatagenai.vector_store.data_classes import HNSW
from teradatagenai.common.constants import CollectionType
from teradataml import create_context, DataFrame, in_schema, remove_context
from teradatasqlalchemy.types import VARCHAR, FLOAT

### Connect to Vantage

**Important Note**: Collection methods are stateless and do not require a database connection for most operations. However, when working with teradataml DataFrames, loading example data, or using certain utility functions, a connection to Vantage is required.

You'll be prompted to provide connection details.

In [ ]:
os.environ['TD_HOST'] = getpass(prompt='hostname: ')
os.environ['TD_USERNAME'] = getpass(prompt='username: ')
os.environ['TD_PASSWORD'] = getpass(prompt='password: ')
os.environ['TD_BASE_URL'] = getpass(prompt='base_url: ')
os.environ['TD_AUTH_MECH'] = 'BASIC'
context = create_context()
print("✓ Connected to Vantage")

Authentication token is generated and set for the session.
✓ Connected to Vantage


### Load Example Data

We'll use datasets with pre-computed embeddings. These embeddings are 1024 dimensions, generated from a previous embedding process.

In [5]:
# Load example datasets with embeddings (1024 dimensions)
# Note: Replace with actual dataset names that have embeddings in your environment
load_data('amazon', 'amazon_reviews_embedded')
load_data('amazon', 'amazon_reviews_embedded_10_alter')

# Create DataFrame for the main dataset
input_data = DataFrame(in_schema('vs_user', 'amazon_reviews_embedded'))
print("✓ Example data with embeddings loaded")

✓ Example data with embeddings loaded


### CHECK AVAILABLE MODELS

In [7]:
CollectionManager.list_available_models()

embedding_models:
                             model_id                      model_name provider  status
0     amazon.titan-embed-text-v1:2:8k      Titan Embeddings G1 - Text   Amazon  ACTIVE
1        amazon.titan-embed-text-v2:0        Titan Text Embeddings V2   Amazon  ACTIVE
2       amazon.titan-embed-image-v1:0  Titan Multimodal Embeddings G1   Amazon  ACTIVE
3         amazon.titan-embed-image-v1  Titan Multimodal Embeddings G1   Amazon  ACTIVE
4             cohere.embed-english-v3                   Embed English   Cohere  ACTIVE
5  cohere.embed-multilingual-v3:0:512              Embed Multilingual   Cohere  ACTIVE
6       cohere.embed-english-v3:0:512                   Embed English   Cohere  ACTIVE
7          amazon.titan-embed-text-v1      Titan Embeddings G1 - Text   Amazon  ACTIVE
8       amazon.titan-embed-g1-text-02        Titan Text Embeddings v2   Amazon  ACTIVE
9                   cohere.embed-v4:0                        Embed v4   Cohere  ACTIVE

chat_models:
           

---
## 2. Initialize Models

For embedding-based collections:
- **Embedding model**: AWS model (for search/ask operations) - we'll pass this during search, not during create
- **Chat completion model**: NVIDIA NIM model

### Initialize Chat Completion Model

We'll initialize the NIM chat model first.

In [31]:
hosted_embedding_model = TeradataAI(api_type = "td_hosted",
                                    model_name = "amazon.titan-embed-text-v2:0")


hosted_chat_model = TeradataAI(api_type = "td_hosted",
                                model_name = "anthropic.claude-3-haiku-20240307-v1:0",
                                )

print("✓ Embedding and Chat models initialized")

✓ Embedding and Chat models initialized


---
## 3. Create Embedding-Based Collection

Now we'll create an embedding-based collection using the `create()` method.

**Important**: For embedding-based collections, we don't pass the embedding model during creation since we're using pre-computed embeddings. We'll pass it during search/ask operations.

**Step 1**: Define column information using `ColumnInfo`  
**Step 2**: Build an `EmbeddingBasedIndex` with the column definitions  
**Step 3**: Initialize a Collection instance  
**Step 4**: Call `create()` with the index and chat model

We'll configure:
- **key_columns**: Unique identifiers for each record
- **data_columns**: Text content (for context)
- **embedding_columns**: Pre-computed embedding vectors
- **metadata_columns**: Additional information to store
- **search_params**: HNSW algorithm configuration

In [9]:
# Step 2: Define column information using ColumnInfo

# Key columns
rev_id_column = ColumnInfo(
    name="rev_id",
    datatype=VARCHAR(50),
    description="Review ID - Primary Key",
    sources=[input_data.rev_id]
)

aid_column = ColumnInfo(
    name="aid",
    datatype=VARCHAR(50),
    description="Author ID - Primary Key",
    sources=[input_data.aid]
)

# Data column (text content)
rev_text_column = ColumnInfo(
    name="rev_text",
    datatype=VARCHAR(64000),
    description="Review text for context",
    sources=[input_data.rev_text]
)

# Embedding column (pre-computed embeddings - 1024 dimensions)
embedding_column = ColumnInfo(
    name="embedding",
    description="Pre-computed embeddings (1024 dimensions)",
    sources=[input_data.embedding]
)

# Metadata columns
prodsummary_column = ColumnInfo(
    name="prodsummary",
    datatype=VARCHAR(4000),
    description="Product summary metadata",
    sources=[input_data.prodsummary]
)

print("✓ Column definitions created")

✓ Column definitions created


In [10]:
# Step 3: Build the EmbeddingBasedIndex using the DataFrame
embedding_index = EmbeddingBasedIndex(
    object_names=input_data,
    key_columns=[rev_id_column, aid_column],
    data_columns=[rev_text_column],
    embedding_columns=[embedding_column],
    metadata_columns=[prodsummary_column]
)

print("✓ EmbeddingBasedIndex defined")

✓ EmbeddingBasedIndex defined


In [11]:
# Step 4: Define HNSW parameters for indexing algorithm
hnsw_params = HNSW(
    metric="COSINE",
    ef_construction=15,
    ef=100
)

print("✓ HNSW parameters defined")

✓ HNSW parameters defined


In [32]:
# Step 5: Initialize Collection and create
collection = Collection(name='collection_amazon_reviews_embeddings')

collection.create(
    type=CollectionType.EMBEDDING_BASED,
    index=embedding_index,
    embedding_model = hosted_embedding_model,  # Pass embedding model for search
    indexing_algorithm=hnsw_params
)

print(f"✓ Collection '{collection.name}' created")

/root/TDGENAI-RC-20.0.0.6/tdg-rc/lib/python3.12/site-packages/teradatagenai/vector_store/collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection create request is accepted and in progress
✓ Collection 'collection_amazon_reviews_embeddings' created


### Check Creation Status

In [27]:
print("\nCollection Status:")
collection.status()


Collection Status:


collection_name,collection_status
collection_amazon_reviews_embeddings,ready


---
## 4. Retrieve Collection Information

Let's explore various methods to get information about our collection.

### Get Collection Details

In [14]:
print("Collection Details:")
collection.get_details()

Collection Details:


collection_name,collection_status,collection_type,collection_description,target_database,permission,last_updated
collection_amazon_reviews_embeddings,ready,EMBEDDING-BASED,None,vs_user,ADMIN,2026-02-19 14:36:32.455392+00:00


### Get Indexes and Embeddings

This method returns the underlying vector store table containing embeddings and metadata.

In [15]:
indexes_embeddings = collection.get_indexes_embeddings()
print("\nIndexes and Embeddings:")
print(f"Shape: {indexes_embeddings.shape}")
indexes_embeddings.head(1)


Indexes and Embeddings:
Shape: (25, 9)


DataBaseName,TableName,TD_ID,rev_id,aid,prodsummary,rev_text,vector_index,vector_index_normalized
vs_user,amazon_reviews_embedded,8,A1TT4CY55WLHAR,000100039X,"textured paper, old-style typography, and leather cover","I have the 1972 version, bought in 1974. The 1972 version originally had a dust jacket but my dust jacket is long gone. This particular rendition has had many re-printings, for a reason: it\'s very popular. The textured paper, old-style typography, and leather cover are better than a plain-old paperback. The size is diminutive, which is perfect for this book.","-0.0267419721931219,0.00441572722047567,0.0264118257910013,0.0567854233086109,-0.0227801986038685,-0.0165073908865452,0.0140312816947699,-0.0835273936390877,0.00462206918746233,0.0245960131287575,-0.00986316613852978,0.0174152981489897,0.00301259872503579,-0.00456016650423408,-0.0498523190617561,-0.0330147817730904,0.00623153988271952,-0.0116377100348473,-0.0364813320338726,0.0567854233086109,-0.046550840139389,-0.0231103468686342,0.00697437254711986,0.00936794467270374,-0.0203866269439459,-0.00833623204380274,0.0485317297279835,0.00792354717850685,0.0409383289515972,-0.0150217255577445,0.0590964592993259,0.0482015796005726,0.00282689067535102,0.00676803011447191,0.068670742213726,0.0828671008348465,-0.00202215532772243,0.0597567558288574,-0.00356972333975136,0.0108123412355781,-0.00697437254711986,0.0272371955215931,-0.00334274652414024,0.0449001044034958,-0.00949175003916025,0.0219548288732767,0.0557949803769588,0.00423001870512962,0.00647915108129382,0.0290530081838369,0.0189009625464678,0.0103583876043558","-0.0267419721931219,0.00441572722047567,0.0264118257910013,0.0567854233086109,-0.0227801986038685,-0.0165073908865452,0.0140312816947699,-0.0835273936390877,0.00462206918746233,0.0245960131287575,-0.00986316613852978,0.0174152981489897,0.00301259872503579,-0.00456016650423408,-0.0498523190617561,-0.0330147817730904,0.00623153988271952,-0.0116377100348473,-0.0364813320338726,0.0567854233086109,-0.046550840139389,-0.0231103468686342,0.00697437254711986,0.00936794467270374,-0.0203866269439459,-0.00833623204380274,0.0485317297279835,0.00792354717850685,0.0409383289515972,-0.0150217255577445,0.0590964592993259,0.0482015796005726,0.00282689067535102,0.00676803011447191,0.068670742213726,0.0828671008348465,-0.00202215532772243,0.0597567558288574,-0.00356972333975136,0.0108123412355781,-0.00697437254711986,0.0272371955215931,-0.00334274652414024,0.0449001044034958,-0.00949175003916025,0.0219548288732767,0.0557949803769588,0.00423001870512962,0.00647915108129382,0.0290530081838369,0.0189009625464678,0.0103583876043558"


---
## 5. Update Collection

Collections can be updated using the `update()` method. Let's demonstrate:
<br/> 1. Adding datasets
<br/> 2. Deleting datasets

### Adding Data

We'll add the `amazon_reviews_10_embeddings` dataset to our existing collection using the `update()` method with `alter_operation='ADD'`.

In [17]:
# Create DataFrame for additional data
additional_reviews_df = DataFrame(in_schema("vs_user", "amazon_reviews_embedded_10_alter"))

# Create index for additional data
additional_data_index = EmbeddingBasedIndex(object_names=additional_reviews_df)

# Add the dataset to collection using update()
collection.update(
    index=additional_data_index,
    alter_operation='ADD',
    update_style='MINOR'
)

print("✓ Dataset added to collection")

Collection update request is accepted and in progress
✓ Dataset added to collection


In [22]:
# Check status after adding
print("\nAdd Dataset Status:")
collection.status()


Add Dataset Status:


collection_name,collection_status
collection_amazon_reviews_embeddings,ready


### Deleting Data

Now we'll remove the dataset we just added using the `update()` method with `alter_operation='DELETE'`.

In [20]:
collection.update(
    index=additional_data_index,
    alter_operation='DELETE',
)
print("✓ Dataset deleted from collection")

Collection update request is accepted and in progress
✓ Dataset deleted from collection


In [21]:
# Check status after deleting
print("Delete Dataset Status:")
collection.status()

Delete Dataset Status:


collection_name,collection_status,retry_after
collection_amazon_reviews_embeddings,update_finishing,60


---
## 6. Search Operations

Now let's perform search operations. **Important**: Since this is an embedding-based collection, we need to pass the embedding model during search and ask operations.

### Similarity Search

Find the most similar documents based on a question. We pass the AWS embedding model with embedding dimenstion of 1024 since the pre computed embeddings are of those dimensions

In [33]:
question = 'Which books are the reviews talking about?'

search_results = collection.similarity_search(
    question=question,
    top_k=5
)

print(search_results)

similar_objects_count:5
similar_objects:
      score DataBaseName                TableName  TD_ID          rev_id         aid                              prodsummary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 rev_text  index_label
0  0.173738      vs_user  amazon_reviews_embedded      5  A26GKZPS079GFF  000100039X  Touches my heart.. again and.. again...                                                                                                                                                                                                              

### Similarity Search by Vector

You can also search using a pre-computed embedding vector directly.

In [34]:
# Search using the vector
vector_search_results = collection.similarity_search_by_vector(
    question='-0.06944,0.080352,0.045963,0.006985,-0.000496,-0.025461,0.045302,-0.028107,0.031248,0.00077,-0.028107,0.016781,-0.023147,-0.068779,-0.07936,-0.030091,0.027611,-0.047616,-0.025461,0.029595,0.024635,-0.025461,-0.029926,0.046294,-0.065142,-0.019013,0.037366,-0.008019,0.065472,-0.054891,-0.009507,0.022816,0.009341,-0.041995,0.022651,0.028603,0.059851,0.047286,0.014467,0.002118,0.016616,-0.009383,-0.001643,0.015624,0.002831,0.005539,0.0248,0.018517,-0.007109,-0.013723,0.029926,0.006903,-0.011325,0.075723,0.009259,0.043648,0.035382,-0.02943,0.023147,-0.036208,-0.017856,-0.032736,0.019013,-0.037035,0.022155,-0.036704,-0.003596,-0.012069,0.021824,0.013805,-0.062827,0.016616,0.008928,-0.04431,-0.019592,-0.002397,0.048608,-0.00341,-0.024139,0.006985,-0.005001,0.002542,0.001777,0.002025,0.026123,0.055883,0.015707,0.014963,0.024304,0.001157,0.042326,-0.004753,-0.044971,0.005373,0.074731,0.002728,-0.028934,0.032736,0.011573,-0.012483,-0.040507,0.040507,0.001736,-0.036539,0.028438,0.053568,0.048278,0.082006,0.011739,0.064811,0.034059,0.062496,-0.013309,-0.065803,0.05456,-0.046624,0.009837,0.005539,-0.015376,0.016947,-0.065472,0.015128,-0.018352,0.062496,0.005539,-0.036208,0.001715,-0.023643,-0.000646,-0.047616,0.035712,0.011325,0.013723,0.07936,-0.010375,-0.021989,0.030091,0.013475,0.038358,-0.034059,-0.068118,0.013475,0.036043,-0.017029,-0.028107,-0.002687,0.00992,-0.001963,-0.04431,-0.009052,-0.088619,-0.016699,-0.027611,0.006861,-0.046624,-0.047286,-0.00744,0.00187,0.004133,-0.008225,-0.018352,0.001405,-0.033067,-0.000858,-0.001705,-0.028107,-0.01984,-0.010251,-0.013888,0.002005,0.046294,0.025461,-0.021163,-0.044971,-0.034886,-0.010747,-0.024304,0.006035,-0.019344,0.001126,-0.018352,0.015045,0.02728,-0.020749,0.000527,0.007523,-0.019427,-0.010499,0.038027,-0.027445,-0.018931,0.021659,0.037035,-0.054891,0.005539,-0.051254,-0.003968,0.011739,-0.041003,0.017029,0.011408,-0.007564,0.051584,0.010499,-0.001788,0.001075,0.032902,0.020997,-0.015624,0.020749,0.038027,0.020749,-0.046955,-0.012069,0.050262,-0.048608,0.028934,-0.074731,0.025461,-0.056875,0.013971,-0.018104,-0.054891,-0.001343,0.013888,0.019427,0.038688,0.057536,-0.011077,0.082006,0.024139,0.033894,0.037366,0.02943,-0.026619,-0.032075,-0.040672,-0.021493,0.001891,-0.013805,-0.005415,0.016451,-0.001963,0.003617,0.003286,0.016533,-0.071424,-0.042987,0.023808,-0.071424,0.008101,0.018683,-0.037862,-0.036208,0.037366,0.006531,0.016203,0.031083,-0.006448,0.008349,-0.045963,0.012648,0.032571,-0.007688,-0.043318,0.025792,0.000889,0.015211,-0.018765,-0.013061,0.040176,-0.035712,0.02232,-0.011243,-0.030091,-0.04431,-0.059851,0.011904,-0.013888,0.060182,0.04431,-0.017691,-0.008721,-0.000509,-0.010375,-0.033563,-0.03968,-0.018104,0.010044,0.020005,-0.024304,0.046955,0.001447,0.017029,-0.011739,-0.042326,0.014219,0.036539,0.028768,-0.008597,-0.005745,0.00868,-0.040011,-0.036704,0.007192,-0.042326,0.061504,0.015789,-0.023147,-0.005373,-0.039184,0.014053,-0.000713,0.003038,-0.009135,0.012483,-0.028107,-0.014549,-0.060512,0.048939,-0.069771,0.000607,0.012648,0.017195,0.018683,0.008804,0.030918,-0.038027,-0.00279,0.027611,0.007068,0.031579,0.029099,-0.004898,0.04431,-0.056875,-0.034886,-0.008267,-0.005911,-0.018517,0.009837,0.0248,0.023147,0.04464,-0.017029,0.016285,-0.0124,0.030091,0.012069,-0.013392,-0.000163,0.038027,0.008721,0.005621,-0.015707,-0.062827,-0.060182,0.026453,0.028107,0.015293,0.014219,-0.047616,0.006076,0.032571,-0.047616,-0.027115,0.046294,-0.006985,-0.027445,0.005167,-0.03191,-0.009383,-0.022981,0.025957,0.001974,0.010127,-0.038854,-0.020253,-0.046955,0.011408,-0.047616,0.006985,-0.058198,0.021989,-0.001674,0.011987,-0.006861,0.043979,0.011408,-0.007853,-0.03439,0.025957,0.013557,0.02976,0.008721,-0.012731,-0.054891,-0.058859,-0.004857,-0.005291,-0.008349,0.007895,-0.020997,0.038854,-0.008349,0.016616,-0.060843,0.040507,0.021824,0.002253,-0.012565,0.006324,0.040011,0.073078,-0.02943,0.014384,-0.031083,0.010995,-0.025131,0.009011,0.015707,0.060182,0.05919,0.017443,0.033563,-0.039184,0.051915,0.040011,-0.021493,0.036208,-0.030091,0.022981,0.056214,0.070432,-0.004071,-0.019013,-0.000377,0.014549,-0.044971,-0.019592,0.013392,0.008308,-0.002563,-0.032902,0.001405,0.013061,0.013061,-0.018269,0.022155,-0.038358,0.048608,0.078699,-0.023477,-0.031744,-0.008308,-0.042326,0.015045,0.072086,0.010333,-0.058859,0.001467,0.040011,-0.011243,0.045302,-0.028107,-0.012731,0.056875,-0.003265,-0.015541,0.007771,0.030091,-0.057206,-0.014632,0.020667,-0.041995,0.030752,0.014053,-0.038027,0.013061,0.012565,0.002366,0.049931,-0.02976,0.015045,0.007564,-0.015459,0.04464,0.005745,-0.029099,-0.013557,-0.005869,0.024304,-0.026784,0.007688,-0.073078,-0.046624,-0.00155,0.000107,0.009259,-0.022155,0.015376,0.003885,-0.080352,0.011408,-0.020749,-0.01612,0.011573,0.055222,0.013557,0.00155,0.000961,0.012069,0.002955,-0.014797,-0.041499,-0.018683,-0.035216,-0.043648,0.103169,-0.007688,-0.024965,-0.005911,-0.012483,-0.024635,0.033398,-0.014053,0.015872,0.013723,0.000314,0.033728,-0.020088,0.016947,-0.061504,-0.046624,0.07407,-0.009383,0.001602,-0.040507,-0.071424,0.000899,-0.010127,-0.024635,-0.005828,0.03968,0.021989,-0.001684,-0.028272,0.035216,-0.046294,-0.002614,-0.027941,-0.020667,0.016368,0.001157,0.005952,0.007523,0.017195,0.038027,0.004402,-0.004505,0.057536,0.042987,-0.028438,-0.033728,0.010747,-0.004629,0.026123,0.014219,-0.003245,-0.028107,0.060182,0.03439,0.016947,-3.8e-05,-0.005229,0.013144,0.042987,0.007275,-0.028768,-0.000734,-0.002687,0.030091,-0.01488,-0.027611,-0.012813,-0.015707,-0.023477,0.001509,0.028934,-0.015376,0.000372,0.005456,-0.041334,0.032571,-0.030422,0.009837,-0.054891,-0.016368,0.005497,0.03224,-0.012648,-0.038027,0.002056,-0.035216,-0.017029,-0.046955,0.035712,0.019013,-0.023477,-0.033894,0.005787,0.021493,0.011491,0.0248,0.026288,0.035712,-0.031248,0.012896,0.025627,0.046955,-0.047947,-0.076054,0.054891,-0.045632,5.1e-05,0.006613,-0.003761,-0.021659,-0.038854,0.031414,-0.028438,-0.013061,-0.05952,-0.01612,-0.018021,0.002997,0.003038,-0.047286,-0.015376,-0.021163,-0.016285,-0.002893,-0.011656,0.021659,0.024469,0.01612,0.09391,0.030918,0.009672,-0.021659,0.056214,-0.04431,0.021824,-0.011325,0.014384,-0.0248,0.043318,0.005249,-0.010664,0.030422,0.060182,-0.006903,-0.038854,-0.000153,0.007936,-0.032075,0.008473,-0.008473,0.021989,0.020749,0.020667,-0.016037,-0.016781,-0.004898,0.00744,0.020088,0.026784,0.020088,0.005022,-0.036208,-0.00186,-0.006861,0.005663,-0.014797,0.052907,-0.013723,0.04431,-0.001963,0.043318,-0.006365,0.002501,-0.011987,0.024304,0.010375,-0.017608,-0.033728,0.018517,0.00092,0.002687,-0.004629,-0.002015,0.041499,0.010664,0.048278,0.011573,-0.065142,0.029926,0.018352,0.007812,-0.03439,0.019592,0.026784,0.0496,-0.007688,0.006531,-0.001457,0.0124,0.016285,0.004505,-0.014963,-0.030091,-0.07936,0.001602,0.013144,-0.026288,-0.00062,0.000297,-0.001225,-0.011408,-0.0124,0.007027,-0.004009,-0.003968,0.029926,0.007895,0.033563,0.013061,0.006696,0.009507,-0.009796,0.05456,-0.048608,0.006076,0.032902,-0.020667,-0.002914,0.000925,0.05423,0.011077,-0.036208,0.008845,-0.030752,-0.05919,0.04431,0.037035,-0.009011,-0.020749,0.007647,0.033894,-0.052246,0.037862,0.041334,0.019344,-0.075723,-0.004795,-0.000889,-0.011325,-0.006944,0.036539,0.021163,0.015211,0.075723,-0.02976,-0.022816,-0.027445,-0.014797,0.068779,0.03472,0.107137,-0.03439,-0.001059,-0.013723,-0.023973,0.046955,-0.000853,-0.05456,-0.025957,-0.044971,-0.0496,0.048608,-0.014219,0.010499,-0.015376,-0.026784,-0.023477,0.042326,0.018104,0.018517,0.046955,0.018269,-0.02976,0.028934,-0.031579,-0.009713,0.010747,-0.023477,0.062496,0.046624,-0.042987,0.070763,0.004898,-1.1e-05,-0.03439,0.001953,0.001788,0.016781,-0.02232,0.028768,0.044971,0.028768,-0.00868,-0.037035,0.001498,-0.02728,-0.006985,0.011656,-0.013557,0.010664,-0.052576,-0.026619,0.002645,0.021328,-0.009383,0.007688,-0.026619,0.045963,-0.037862,-0.024139,0.06448,-0.060843,0.045302,0.070432,-0.013805,0.023973,-0.066795,0.095233,0.006159,0.009011,0.04183,-0.065142,0.026123,-0.007068,-0.02232,0.016947,0.038027,0.014384,0.015211,-0.050262,0.038358,-0.008804,0.021659,0.03439,0.023973,0.00248,0.023477,0.002366,-0.003451,0.002459,0.095894,0.004898,-0.001059,0.034886,0.006944,0.005993,0.05952,-0.021328,0.005373,-0.037035,-0.001297,0.051584,-0.0496,0.028934,0.018021,0.042987,0.003989,-0.03191,0.043318,0.036043,0.017029,0.009383,-0.001405,-0.084651,-0.003079,-0.011325,0.004175,0.01488,-0.012483,0.041995,-0.018765,-0.011243,-0.001591,0.024635,0.01984,0.006241,0.009383,0.003761,0.04927,0.011325,0.020667,-0.022816,-0.0496,-0.014715,0.028107,0.001225,0.012152,-0.013888,-0.058198,0.03935,0.024635,0.072416,-0.048939,0.020336,0.015128,0.004237,-0.008597,0.031414,-0.040507,-0.018765,-0.011325,0.056544,0.029595,0.001044,-0.015789,0.05423,0.032902,-0.0031,-0.010333,-0.022816,-0.025461,0.006241,-0.000273,-0.011987,-0.038027,-0.061174,0.003865,-0.02943,0.012731,0.06448,-0.040507,0.011408,0.047947,0.015128,-0.011739,0.021659,0.012069,0.020997,-0.011325,0.003637,0.014549,-0.015045,0.011077,-0.008184,0.005869,-0.037862,-0.000806,0.018517,0.028603,0.00992,-0.000245,0.005249,-0.005084,-0.000692,-0.00094,-0.019179,-0.000625,-0.009135,-0.002811,-0.018104,-0.060182,-0.0248,0.000605,0.017856,0.005022,-0.017443,0.014384,-0.010127,-0.007523,0.041003,0.033563,-0.037366,0.003927,0.00806,-0.048278,0.016533,-0.021989,-0.009176,0.019013,0.022485,-0.005332,-0.026123,-0.014632,0.023973,-0.022155,0.016947,-0.020088,0.008184,-0.021493,0.027941,0.073408,0.03224,0.018104,-0.01736,-0.007275,0.031414,-0.007357,-0.04464,0.045302,-0.010664,-0.016203,0.010375,0.004567,0.0124,-0.009011,-0.010457',  # Pass the embedding vector directly
    top_k=5
)

print("\nSimilarity Search by Vector Results:")
print(vector_search_results)


Similarity Search by Vector Results:
similar_objects_count:5
similar_objects:
      score DataBaseName                TableName  TD_ID          rev_id         aid                                              prodsummary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 rev_text  index_label
0  0.202211      vs_user  amazon_reviews_embedded     11  A2X4HE21JTAL98  000100039X                                                 Flawless  Anything I\'ve read by Gibran is, in my mind, flawless. This, the most famous of his works, is no exception. It is simple, yet deep; h

### Prepare Response

Generate a natural language response based on similarity search results.

In [36]:
question = 'What are customers saying about the quality of the books?'
prompt = 'Provide a concise summary of customer feedback. Format the response in a conversational way.'

# Then prepare response based on results
response = collection.prepare_response(
    question=question,
    similarity_results=search_results,
    chat_model = hosted_chat_model,  # Pass the chat model for response generation
    prompt=prompt
)

print(f"\nQuestion: {question}\n")
print("Response:")
print(response)


Question: What are customers saying about the quality of the books?

Response:
Based on the customer reviews, it seems the customers are very positive about the quality and content of the books. Here's a summary of the key points:

- Customers describe the books as "amazing", "beautiful", and "flawless". They seem to really appreciate the depth, wisdom, and inspirational nature of the author's work.

- Several customers mention that the books are "readable" and easy to get through, while still containing profound and thought-provoking content.

- Customers highlight how the books have impacted them personally, with one saying it "touches my heart again and again" and another describing it as "one of my favorite reads of all time."

- There is a sense that the books provide a valuable guide or perspective on major life decisions and approaches to living. Customers find the author's outlook on life to be "inspiring" and something they don't want to be without.

Overall, the customer fee

### Ask (Combined Search + Response)

The `ask()` method combines similarity search and response generation in a single call. We pass the embedding model here as well.

In [37]:
custom_prompt = """List reviews about the books. Only provide information present in the data.
Format results like this:
- Review ID: 
- Product Summary: 
- Review: 
"""

question = 'Are there any reviews saying that the books are inspiring?'

response = collection.ask(
    question=question,
    chat_model = hosted_chat_model,  # Pass the chat model for response generation
    prompt=custom_prompt,
    top_k=5
)

print(f"\nQuestion: {question}\n")
print("Response:")
print(response)


Question: Are there any reviews saying that the books are inspiring?

Response:
Yes, there are a few reviews that mention the books being inspiring:

- Review ID: A10000012B7CGYKOMPQ4L
Product Summary: Wonderful!
Review: Spiritually and mentally inspiring! A book that allows you to question your morals and will help you discover who you really are!

- Review ID: A2I35JB67U20C0
Product Summary: Everyone should have this book
Review: You see, if you have ideas and you do not realize them, then they are nothing, and if your ideas do not come from love and joy, then they are bitter and what they produce will be bitter, but if they flow from love and joy, then their realization will be love and joy, thus work is love made visible. He says it WAY better than I do and says much more in just a few paragraphs. But as you can see from the example, what he says is not religious, but positive ways of looking at things.


---
## 7. Alternative Methods: Embedding Management

In addition to the `update()` method, teradatagenai provides convenient methods specifically designed for embedding-based collection management:

- **`from_embeddings()`**: Create a collection directly from pre-computed embeddings
- **`add_embeddings()`**: Add tables/DataFrames with embeddings to an existing collection
- **`delete_embeddings()`**: Remove tables/DataFrames from a collection

These methods provide a more intuitive API when working specifically with embedding-based operations.

### Create Collection with from_embeddings()

The `from_embeddings()` class method provides a streamlined way to create a collection directly from datasets with pre-computed embeddings.

In [38]:
# Define key column
rev_id_column_alt = ColumnInfo(
    name="rev_id",
    datatype=VARCHAR(50),
    description="Review ID - Primary Key",
    sources=["amazon_reviews_embedded.rev_id"]
)

# Define data column (original text for reference)
rev_text_column_alt = ColumnInfo(
    name="rev_text",
    datatype=VARCHAR(64000),
    description="Review text content",
    sources=["amazon_reviews_embedded.rev_text"]
)

# Define embedding column (pre-computed vectors)
embedding_column_alt = ColumnInfo(
    name="embedding",
    description="Pre-computed embedding vectors",
    sources=["amazon_reviews_embedded.embedding"]
)

# Create the collection using from_embeddings
collection_alt = Collection.from_embeddings(
    name="collection_embedded_alt",
    data="amazon_reviews_embedded",
    key_columns=rev_id_column_alt,
    data_columns=rev_text_column_alt,
    embedding_columns=embedding_column_alt,
    description="Embedding-based collection created from pre-computed embeddings"
)

print(f"✓ Collection '{collection_alt.name}' created using from_embeddings()")


/root/TDGENAI-RC-20.0.0.6/tdg-rc/lib/python3.12/site-packages/teradatagenai/vector_store/collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection create request is accepted and in progress
✓ Collection 'collection_embedded_alt' created using from_embeddings()


In [43]:
collection_alt.status()

collection_name,collection_status,retry_after
collection_embedded_alt,update_finishing,60


### Add Embeddings with add_embeddings()

Use `add_embeddings()` to add more tables/DataFrames with pre-computed embeddings to an existing collection. This method can also create a new collection if it doesn't exist.

In [41]:
# Add additional embeddings to the collection
collection_alt.add_embeddings(
    data="amazon_reviews_embedded_10_alter",
    update_style="MINOR"
)

print("✓ Embeddings added using add_embeddings()")


Collection update request is accepted and in progress
✓ Embeddings added using add_embeddings()


In [44]:
collection_alt.status()

collection_name,collection_status
collection_embedded_alt,ready


### Delete Embeddings with delete_embeddings()

Remove specific tables/DataFrames from an existing embedding-based collection using `delete_embeddings()`.

In [45]:
# Delete the embeddings we added earlier
collection_alt.delete_embeddings(
    data="amazon_reviews_embedded_10_alter"
)

print("✓ Embeddings deleted using delete_embeddings()")
collection_alt.status()

Collection update request is accepted and in progress
✓ Embeddings deleted using delete_embeddings()


collection_name,collection_status,retry_after
collection_embedded_alt,update_repreparing_input,60


### Cleanup Alternative Collections

Let's clean up the collections created in this section.

In [ ]:
# Destroy the collections created in this section
collection_alt.destroy()


print("✓ Alternative collections destroyed")

✓ Alternative collections destroyed


In [29]:
collection.destroy()
print(f"✓ Collection '{collection.name}' destroyed")

Collection destroy request is accepted and in progress
✓ Collection 'collection_amazon_reviews_embeddings' destroyed


### Remove Database Connection

In [48]:
remove_context()
print("✓ Database connection removed")

✓ Database connection removed


---
## Summary

In this notebook, you learned how to:

✅ Set up your environment and connect to Teradata Vantage  
✅ Load example data with pre-computed embeddings (1024 dimensions)  
✅ Initialize embedding (AWS) and chat (NIM) models using TeradataAI  
✅ Create an embedding-based collection using the `create()` method  
✅ Retrieve collection details, model info, and embeddings  
✅ Update collections using the `update()` method with ADD and DELETE operations  
✅ Perform similarity search operations (passing embedding model during search)  
✅ Generate natural language responses using `prepare_response()` and `ask()`  
✅ Search by vector directly using `similarity_search_by_vector()`  
✅ Use alternative embedding management methods (`from_embeddings()`, `add_embeddings()`, `delete_embeddings()`)  
✅ Clean up resources properly  
